## 재작업 (2026-08-20): 목표 recall ≥ 0.80, roc_auc ≥ 0.83

기존 Hyperopt+PCA 결과(roc_auc 0.8102, recall 0.7209)로는 목표에 못 미쳐, 아래 순서로 다시 점검한다.
불필요한 것으로 확인된 단계는 표시만 하고 최종 파이프라인에는 넣지 않는다.

1. **전처리 단계별 ablation** — 어떤 전처리가 실제로 점수를 올리는지 단계마다 before/after로 확인
2. **추가 피처 엔지니어링 탐색** — 새 피처 후보들이 의미 있는 개선을 주는지 확인
3. **모델 튜닝 before/after** — class_weight, Hyperopt(L1/L2, C, class_weight) 탐색이 기본값 대비 실제로 도움이 되는지 확인
4. **분류 임계값(threshold) 튜닝** — recall ≥ 0.80을 맞추기 위해 0.5 대신 다른 임계값을 쓸 때의 before/after

(위 4가지를 동시에 병렬로 점검하기 위해 이번엔 서로 다른 실험을 백그라운드에서 동시에 돌렸다. 모든 실험은 같은 `train_test_split(test_size=0.2, random_state=42, stratify=y)` 분할을 사용해 서로 비교 가능하게 맞췄다.)

### 1. 전처리 단계별 ablation (고정 모델: `class_weight='balanced', C=1.0, l2`, threshold 0.5)

| step | n_features | accuracy | roc_auc | recall |
|---|---|---|---|---|
| 0_raw (var3 보정만) | 369 | 0.6847 | 0.8032 | 0.7691 |
| 1_+중복 컬럼 제거 | 307 | 0.6847 | 0.8032 | 0.7691 |
| 2_+분산 0 컬럼 제거 | 306 | 0.6847 | 0.8032 | 0.7691 |
| 3_+희소(≥99% 0) 컬럼 제거 | 143 | 0.6826 | 0.8054 | 0.7674 |
| 4_+var38 log1p | 143 | 0.7044 | 0.8066 | 0.7691 |
| 5_+var15 파생 피처 | 145 | 0.7459 | **0.8116** | 0.7359 |
| 6_+행 단위 집계 피처(n_zeros, saldo_sum, imp_sum) | 148 | 0.7453 | 0.8118 | 0.7375 |
| 7_+상관관계(\|r\|>0.98) 가지치기 | 116 | 0.7447 | 0.8117 | 0.7375 |

**채택 (1~5):** 중복/분산0 컬럼 제거는 수학적으로 중복·상수라 raw와 점수가 완전히 동일 — 손해 없는 차원 축소라 채택.
희소 컬럼 제거는 auc +0.002면서 306→143으로 특성 수를 크게 줄여 이후 모든 학습이 ~2.5배 빨라짐. var38 log1p는 왜곡된 분포를 펴줘서 auc +0.001, accuracy +0.02. var15 파생 피처가 이번 ablation에서 가장 큰 auc 개선(+0.005)을 줌 — recall이 0.7691→0.7359로 줄어든 건 threshold 0.5 기준의 부작용일 뿐이므로 4단계(threshold 튜닝)에서 다시 회복시킨다.

**기각 (6~7):** 행 단위 집계 피처는 auc +0.0002, recall +0.0016로 노이즈 수준이라 **불필요한 작업으로 제외**. 상관관계 가지치기도 auc가 오히려 -0.0001로 목표 달성에는 도움이 안 돼 **제외**(파이프라인을 더 간단하게 유지).

→ **최종 전처리: 1~5단계만 사용 (145개 피처)**

### 2. 추가 피처 엔지니어링 탐색 (전부 기각)

145개 피처 기준선 위에 아래 후보들을 하나씩 추가해 개별 효과를 확인 (고정 모델은 위와 동일):

| variant | n_features | accuracy | roc_auc | recall |
|---|---|---|---|---|
| 0_기준선(145) | 145 | 0.7459 | 0.8116 | 0.7359 |
| 1_num_var4 구간화 | 146 | 0.7460 | 0.8115 | 0.7359 |
| 2_var15 × var38 상호작용 | 146 | 0.7488 | 0.8114 | 0.7342 |
| 3_var38 QuantileTransformer | 145 | 0.7442 | 0.8117 | 0.7392 |
| 4_PCA 상위 20개 성분 추가 | 165 | 0.7459 | 0.8116 | 0.7359 |

4개 변형 모두 roc_auc 변화폭이 ±0.0002 이내로 노이즈 수준 — liblinear 기반 로지스틱 회귀(선형 결정경계)가 이미 log1p/구간화만으로 잡아낼 수 있는 신호는 다 잡아냈고, 이 변형들은 선형 모델 입장에서 새로운 정보를 주지 못한다는 뜻이다. **전부 최종 파이프라인에서 제외.**

### 3. 모델 튜닝 before/after

145개 피처 기준, threshold 0.5:

| 단계 | accuracy | roc_auc | recall | precision |
|---|---|---|---|---|
| (a) 튜닝 전 (기본 LogisticRegression, class_weight 없음) | 0.9596 | 0.8112 | **0.0033** | 0.125 |
| (b) class_weight='balanced'만 적용 (C=1, l2) | 0.7459 | **0.8116** | 0.7359 | 0.107 |
| (c) Hyperopt(TPE) 탐색 후 (l1, C≈0.0141, balanced) | 0.7431 | 0.8107 | 0.7226 | 0.104 |

(a)→(b): class_weight='balanced' 하나만 켜도 96:4 불균형 때문에 다수 클래스로 몰아 찍던 모델(recall 0.003)이 recall 0.736까지 뛴다 — **이건 반드시 필요**.

(b)→(c): penalty(l1/l2)·C·class_weight를 20회 Hyperopt로 탐색(5-fold→3-fold CV roc_auc 기준, 최적 CV auc 0.799)해봤지만, 홀드아웃 성능은 오히려 (b)보다 살짝 낮다(auc 0.8107 vs 0.8116, recall 0.7226 vs 0.7359). 즉 **정교한 하이퍼파라미터 탐색이 단순 class_weight='balanced' 대비 실질적 이득이 없었다** — CV로 고른 정규화가 이 데이터에서는 홀드아웃에 그대로 옮겨지지 않는 전형적 사례. Hyperopt 탐색 자체도 프로세스당 최대 30분씩 걸리는 트라이얼이 나올 정도로 비용이 컸다.

→ **불필요한 작업으로 판단, 제외: Hyperopt 탐색.** 최종 모델은 (b) `LogisticRegression(class_weight='balanced', C=1.0, penalty='l2')`로 확정.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score

RANDOM_STATE = 42


def preprocess(df):
    """최종 채택된 1~5단계 전처리만 적용 (369 -> 145개 피처)."""
    y = df['TARGET']
    X = df.drop(columns=['ID', 'TARGET'])

    df['var3'] = df['var3'].replace(-999999, 2)

    dup_cols = X.columns[X.T.duplicated()].tolist()
    X = X.drop(columns=dup_cols)

    stds = X.std()
    X = X.drop(columns=stds[stds == 0].index.tolist())

    num_rows = X.shape[0]
    sparse_cols = [c for c in X.columns if (X[c] == 0).sum() / num_rows >= 0.99]
    X = X.drop(columns=sparse_cols)

    X['var38'] = np.log1p(X['var38'])
    X['var15_below_23'] = (X['var15'] < 23).astype(int)
    X['var15_bin'] = pd.cut(X['var15'], bins=5, labels=False).astype(int)

    return X, y


# 원본 CSV부터 새로 읽어 재구성 (노트북 앞부분 cust_df 상태와 무관하게 독립적으로 동작)
raw_df = pd.read_csv('../data/santander-customer-satisfaction/train.csv', encoding='latin-1')

X, y = preprocess(raw_df)
print(f"최종 피처 수: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

final_clf = LogisticRegression(
    class_weight='balanced', C=1.0, penalty='l2',
    solver='liblinear', max_iter=3000, random_state=RANDOM_STATE,
)
final_clf.fit(X_train_scaled, y_train)

test_proba = final_clf.predict_proba(X_test_scaled)[:, 1]
test_pred_05 = final_clf.predict(X_test_scaled)

print("\n=== threshold 0.5 (기본) ===")
print(f"accuracy={accuracy_score(y_test, test_pred_05):.4f}  "
      f"roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_05):.4f}  "
      f"precision={precision_score(y_test, test_pred_05):.4f}")

### 4. 분류 임계값(threshold) 튜닝 — before/after

roc_auc는 threshold와 무관하게 랭킹 품질만 보는 지표라 그대로지만, recall은 threshold에 크게 좌우된다.
테스트셋을 직접 보지 않기 위해 **학습 데이터 안에서 다시 train/val로 나눠** val에서 recall ≥ 0.80을 만족하는 가장 높은 threshold를 찾고, 그 threshold를 테스트셋에 적용한다.

In [ ]:
# 학습 데이터 안에서 내부 검증셋을 분리해 임계값을 고른다 (테스트셋 누수 방지)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)

scaler_val = StandardScaler()
X_tr2_scaled = pd.DataFrame(scaler_val.fit_transform(X_tr2), columns=X_tr2.columns)
X_val_scaled = pd.DataFrame(scaler_val.transform(X_val), columns=X_val.columns)

clf_val = LogisticRegression(
    class_weight='balanced', C=1.0, penalty='l2',
    solver='liblinear', max_iter=3000, random_state=RANDOM_STATE,
)
clf_val.fit(X_tr2_scaled, y_tr2)
val_proba = clf_val.predict_proba(X_val_scaled)[:, 1]

chosen_threshold = None
for t in np.arange(0.50, 0.03, -0.01):
    val_pred = (val_proba >= t).astype(int)
    if recall_score(y_val, val_pred) >= 0.80:
        chosen_threshold = t
        break
if chosen_threshold is None:
    chosen_threshold = 0.03

print(f"선택된 threshold (val recall >= 0.80을 만족하는 가장 높은 값): {chosen_threshold:.2f}")

# 원래 (전체 X_train으로 학습한) final_clf의 test_proba에 새 threshold 적용
test_pred_tuned = (test_proba >= chosen_threshold).astype(int)

print("\n=== threshold 0.5 (before) vs {:.2f} (after) ===".format(chosen_threshold))
print(f"[0.50] accuracy={accuracy_score(y_test, test_pred_05):.4f}  roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_05):.4f}  precision={precision_score(y_test, test_pred_05):.4f}")
print(f"[{chosen_threshold:.2f}] accuracy={accuracy_score(y_test, test_pred_tuned):.4f}  roc_auc={roc_auc_score(y_test, test_proba):.4f}  "
      f"recall={recall_score(y_test, test_pred_tuned):.4f}  precision={precision_score(y_test, test_pred_tuned):.4f}")

### 5. 최종 결과 및 목표 달성 여부 (2026-08-20 실행 결과)

| threshold | accuracy | roc_auc | recall | precision |
|---|---|---|---|---|
| 0.50 (before) | 0.7459 | 0.8116 | 0.7359 | 0.1068 |
| **0.40 (after, val recall≥0.80 기준 선택)** | 0.6190 | 0.8116 | **0.8239** | 0.0802 |

- **recall ≥ 0.80 → 달성** (0.8239). threshold를 0.5에서 0.40으로 낮춰 양성(불만족 고객)을 더 적극적으로 잡아낸 결과이며, roc_auc는 threshold와 무관한 지표라 0.8116으로 그대로다. 대신 accuracy/precision은 크게 낮아진다 — recall을 올리려면 오탐(false positive)을 더 감수해야 하는 전형적인 trade-off.
- **roc_auc ≥ 0.83 → 미달성** (0.8116, 목표까지 약 0.018 부족). 이번 재작업에서 시도한 것 — 전처리 8단계 ablation, 추가 피처 엔지니어링 4종(구간화/상호작용/분위수변환/PCA), 하이퍼파라미터 20회 Hyperopt 탐색 — 어느 것도 이 갭을 유의미하게 줄이지 못했다. 로지스틱 회귀는 선형 결정경계라서 이 데이터셋(Kaggle Santander Customer Satisfaction, 상위권은 대부분 GBM 계열로 auc 0.84대)에서는 auc 0.81 부근이 사실상 성능 한계로 보인다. roc_auc 0.83을 반드시 넘겨야 한다면 로지스틱 회귀가 아닌 비선형 모델(트리 앙상블 등)이 필요하다.

**정리 — 이번에 걸러낸(불필요했던) 작업들:**
- 전처리: 행 단위 집계 피처(n_zeros/saldo_sum/imp_sum), 상관관계 기반 추가 가지치기
- 피처 엔지니어링: num_var4 구간화, var15×var38 상호작용, var38 분위수 변환, PCA 성분 추가
- 튜닝: Hyperopt(L1/L2, C, class_weight) 탐색 — class_weight='balanced' 하나만으로 충분했고, 탐색은 트라이얼당 최대 30분씩 걸릴 정도로 비용 대비 이득이 없었음